# Walmart Store Sales Forecasting — ARIMA

**ლოგირება:** Weights & Biases — პროექტი `walmart-sales-forecasting`/`ML-Final`, group `ARIMA_Training`.

ARIMA **უნივარიატულია და თითო სერიაზე ცალკე ეტრენინგება** — არ იზიარებს წონებს სერიებს შორის (განსხვავებით DLinear/N-BEATS-ის channel-shared მიდგომისგან) და **არ იღებს ეგზოგენურ ცვლადებს** ამ ბაზისურ ფორმაში (ეს SARIMAX-ის საქმეა). ორდერის შერჩევა კლასიკურად ხდება Box-Jenkins მეთოდოლოგიით (ACF/PACF გრაფიკები) ან ავტომატურად, `pmdarima.auto_arima`-თი (AIC-ის მინიმიზაცია).


**გამოწვევა 3 331 სერიაზე:** თითო (Store, Dept) წყვილზე ცალკე ARIMA fit საკმაოდ ძვირია. ამიტომ:
1. ორდერს ვირჩევთ `auto_arima`-ით სერიების **representative sample**-ზე (არა ყველაზე), AIC-ის მიხედვით.
2. არჩეულ ორდერს ვამოწმებთ **ყველა** სერიაზე, `statsmodels.SARIMAX(order=..., seasonal_order=(0,0,0,0))` სწრაფი `fit()`-ით — fit-ი, რომელიც ვერ დაკონვერგირდება/ვერ დაფიტდება, fallback საშუალოზე გადადის (იგივე ლოგიკა, რაც DLinear-ის pipeline-ს აქვს ახალი/მოკლე სერიებისთვის).

Run-ების სტრუქტურა:
1. `ARIMA_Preprocessing` — Date × (Store, Dept) მატრიცის აგება (იგივე, რაც გუნდის სხვა notebook-ებში)
2. `ARIMA_Order_Selection` — `auto_arima` sample-ზე, საუკეთესო `(p,d,q)` კანდიდატების შერჩევა
3. `ARIMA_candidate_{i}` — 2-3 კანდიდატი ორდერი, ყველა სერიაზე fit, შეფასება გუნდის საერთო holdout-ზე WMAE-თი
4. `ARIMA_Final_Pipeline` — საბოლოო მოდელი მთელ ისტორიაზე, 39-კვირიანი ჰორიზონტით; ინახება W&B Artifact-ად, როგორც end-to-end pipeline, რომელიც პირდაპირ raw test-ზე ეშვება

In [1]:
!pip install -q wandb pmdarima statsmodels cloudpickle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 8.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import wandb

wandb.login()

WANDB_PROJECT = 'ML-Final'
GROUP = 'ARIMA_Training'
NB_VERSION = 'v1'

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amakh23 (dshon23-free-university-of-tbilisi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
import numpy as np
import pandas as pd

BASE = '/content/drive/MyDrive/ML Final/Data/Raw/walmart-recruiting-store-sales-forecasting/'

train = pd.read_csv(BASE + 'train.csv')
test = pd.read_csv(BASE + 'test.csv')
features = pd.read_csv(BASE + 'features.csv')
stores = pd.read_csv(BASE + 'stores.csv')

for d in (train, test, features):
    d['Date'] = pd.to_datetime(d['Date'])

RAW_COLS = ['Store', 'Dept', 'Date', 'IsHoliday']
VAL_CUTOFF = '2012-08-17'


def wmae(y_true, y_pred, is_holiday):
    """Competition metric: MAE, სადაც სადღესასწაულო კვირებს წონა 5 აქვთ."""
    w = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)


print(train.shape, test.shape, features.shape, stores.shape)

(421570, 5) (115064, 4) (8190, 12) (45, 3)


## Run 1 — Preprocessing

ARIMA-ც უნივარიატულ ფანჯრებზე მუშაობს (თითო სერია ცალკე), ამიტომ იგივე **Date × (Store, Dept)** wide-მატრიცა გვჭირდება, რაც DLinear/N-BEATS/PatchTST notebook-ებში:
- უარყოფითი sales → 0 (გუნდის EDA-ს გადაწყვეტილება)
- სერიები, რომლებიც გვიან იწყება ან გამოტოვებული კვირები აქვთ → 0-ით ივსება
- **განსხვავება DL მოდელებისგან: ARIMA-ს არ სჭირდება ნორმალიზაცია** (თითო სერია ცალკე ეტრენინგება საკუთარი მასშტაბით, ამიტომ cross-series სტანდარტიზაცია არ არის საჭირო)

In [5]:
run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='ARIMA_Preprocessing',
                 job_type='preprocessing', config={'nb_version': NB_VERSION})

long = train.copy()
long['Weekly_Sales'] = long['Weekly_Sales'].clip(lower=0)

wide = (long.pivot_table(index='Date', columns=['Store', 'Dept'],
                         values='Weekly_Sales', fill_value=0.0)
             .sort_index())
holiday_by_date = long.drop_duplicates('Date').set_index('Date')['IsHoliday'].sort_index()

train_wide = wide[wide.index < VAL_CUTOFF]
val_wide = wide[wide.index >= VAL_CUTOFF]
val_holiday = holiday_by_date.reindex(val_wide.index)

n_weeks, n_series = wide.shape
missing_pct = 100 * (long.pivot_table(index='Date', columns=['Store', 'Dept'],
                                      values='Weekly_Sales').isna().sum().sum()) / (n_weeks * n_series)

run.log({'n_weeks': n_weeks, 'n_series': n_series, 'missing_cell_pct': missing_pct,
         'train_weeks': len(train_wide), 'val_weeks': len(val_wide)})
run.finish()

print(f'weeks x series: {wide.shape} | missing cells: {missing_pct:.1f}% | val weeks: {len(val_wide)}')

missing_cell_pct,▁
n_series,▁
n_weeks,▁
train_weeks,▁
val_weeks,▁
missing_cell_pct,11.49679
n_series,3331
n_weeks,143
train_weeks,132
val_weeks,11


weeks x series: (143, 3331) | missing cells: 11.5% | val weeks: 11


## Run 2 — Order Selection (`auto_arima` sample-ზე)

3 331 სერიაზე `auto_arima` (grid + AIC search) ერთდერთ ერთზე რამდენიმე წამი მაინც სჭირდება — სრულ სეტზე გაშვება საათობით გაგრძელდებოდა, დავალების მითითების საწინააღმდეგოდ („ტრენინგზე დიდი დრო არ დახარჯოთ"). ამიტომ:

- ვირჩევთ **წონით-რეპრეზენტატულ სემფლს** — 40 შემთხვევითი (Store, Dept) სერია, სტრატიფიცირებული საშუალო გაყიდვების კვარტილით (რომ მცირე და დიდი სერიები ორივე იყოს წარმოდგენილი).
- თითოეულზე ვუშვებთ `auto_arima(seasonal=False, max_p=3, max_q=3, max_d=2)`-ს და ვკრებთ არჩეულ `(p,d,q)`-ებს.
- **majority vote** + საშუალო AIC გვაძლევს 2-3 კანდიდატ ორდერს შემდეგი ეტაპისთვის.

In [6]:
from pmdarima import auto_arima
from collections import Counter

rng = np.random.RandomState(42)
series_mean = train_wide.mean(axis=0)
quartile = pd.qcut(series_mean, 4, labels=False, duplicates='drop')

sample_cols = []
for q in sorted(quartile.dropna().unique()):
    pool = series_mean[quartile == q].index.tolist()
    picks = rng.choice(len(pool), size=min(10, len(pool)), replace=False)
    sample_cols += [pool[i] for i in picks]
sample_cols = list(dict.fromkeys(sample_cols))[:40]

run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='ARIMA_Order_Selection',
                 job_type='order_selection', config={'sample_size': len(sample_cols)})

orders_found = []
for col in sample_cols:
    y = train_wide[col].values
    try:
        m = auto_arima(y, seasonal=False, max_p=3, max_q=3, max_d=2,
                       suppress_warnings=True, error_action='ignore')
        orders_found.append(m.order)
    except Exception:
        continue

order_counts = Counter(orders_found)
top_orders = [o for o, _ in order_counts.most_common(3)]

run.log({'n_fit_ok': len(orders_found)})
run.summary['top_orders'] = str(top_orders)
run.summary['order_counts'] = str(dict(order_counts))
run.finish()

print('most common orders (auto_arima, sample of', len(sample_cols), 'series):')
print(order_counts.most_common(5))
print('candidates for next step:', top_orders)

n_fit_ok,▁
n_fit_ok,40
order_counts,"{(0, 0, 2): 3, (0, 0..."
top_orders,"[(0, 1, 1), (1, 0, 0..."


most common orders (auto_arima, sample of 40 series):
[((0, 1, 1), 5), ((1, 0, 0), 5), ((2, 0, 0), 5), ((1, 1, 1), 4), ((0, 0, 2), 3)]
candidates for next step: [(0, 1, 1), (1, 0, 0), (2, 0, 0)]


## Run 3 — კანდიდატების შედარება (ყველა სერიაზე)

ავირჩიეთ ორდერების top-3 კანდიდატი წინა ეტაპიდან (თუ `auto_arima` სემფლზე რაიმე მიზეზით ცარიელი დაბრუნდა, ვამატებთ სტანდარტულ `(1,1,1)`-ს fallback-ად). თითოეულ კანდიდატს ვუსვამთ **ყველა** სერიაზე (`statsmodels.SARIMAX(order=order, seasonal_order=(0,0,0,0))` — ეს არის ARIMA, seasonal ნაწილის გარეშე), ვპროგნოზირებთ validation-ის სიგრძეზე და ვზომავთ WMAE-ს. Fit, რომელიც ვერ დაკონვერგირდება ან სერიას საკმარისი ისტორია არ აქვს → fallback (Store-Dept საშუალო).

In [7]:
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')

CANDIDATE_ORDERS = top_orders if top_orders else [(1, 1, 1)]
if (1, 1, 1) not in CANDIDATE_ORDERS:
    CANDIDATE_ORDERS.append((1, 1, 1))
CANDIDATE_ORDERS = CANDIDATE_ORDERS[:3]

H = len(val_wide)
sd_fallback = train_wide.mean(axis=0)


def fit_forecast_all_series(order, wide_train, horizon):
    """თითო სერიაზე ცალკე ARIMA fit + h-ნაბიჯიანი პროგნოზი. აბრუნებს (n_weeks_h, n_series) მასივს."""
    preds = np.zeros((horizon, wide_train.shape[1]))
    n_fallback = 0
    for j, col in enumerate(wide_train.columns):
        y = wide_train[col].values
        try:
            model = SARIMAX(y, order=order, seasonal_order=(0, 0, 0, 0),
                            enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False, maxiter=50)
            fc = fit.forecast(steps=horizon)
            preds[:, j] = np.clip(fc, 0, None)
        except Exception:
            preds[:, j] = sd_fallback[col]
            n_fallback += 1
    return preds, n_fallback


results = {}
for order in CANDIDATE_ORDERS:
    run = wandb.init(project=WANDB_PROJECT, group=GROUP, name=f'ARIMA_candidate_{order}',
                     job_type='candidate', reinit=True,
                     config={'order': order, 'nb_version': NB_VERSION})

    preds, n_fallback = fit_forecast_all_series(order, train_wide, H)
    pred_df = pd.DataFrame(preds, index=val_wide.index, columns=val_wide.columns)

    y_true = val_wide.values.ravel()
    y_pred = pred_df.values.ravel()
    is_hol = np.repeat(val_holiday.values, val_wide.shape[1])
    score = wmae(y_true, y_pred, is_hol)

    run.log({'val_wmae': score, 'n_fallback_series': n_fallback})
    run.summary['val_wmae'] = score
    run.finish()

    results[order] = score
    print(f'order={order} -> val WMAE={score:.1f} (fallback series: {n_fallback})')

best_order = min(results, key=results.get)
best_val = results[best_order]
BEST_CFG = {'order': best_order}
print('\nBEST:', BEST_CFG, '-> val WMAE', best_val)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


n_fallback_series,▁
val_wmae,▁
n_fallback_series,0
val_wmae,1914.19482


order=(0, 1, 1) -> val WMAE=1914.2 (fallback series: 0)


n_fallback_series,▁
val_wmae,▁
n_fallback_series,0
val_wmae,2714.79653


order=(1, 0, 0) -> val WMAE=2714.8 (fallback series: 0)


n_fallback_series,▁
val_wmae,▁
n_fallback_series,0
val_wmae,2323.54554


order=(2, 0, 0) -> val WMAE=2323.5 (fallback series: 0)

BEST: {'order': (0, 1, 1)} -> val WMAE 1914.1948218719908


## Run 4 — Final Pipeline

საბოლოო მოდელი: საუკეთესო ორდერი ვუსვამთ **მთელ ისტორიაზე** (train სრულად, validation cutoff-ის გარეშე) და ვპროგნოზირებთ Kaggle-ის ტესტის სრულ 39-კვირიან ჰორიზონტზე. Pipeline იგივე ლოგიკით მუშაობს, როგორც სხვა მოდელებში - მზა პროგნოზების long-ცხრილი + fallback საშუალოები (Store-Dept → Dept → global mean) train-ში უნახავი ან ვერ-დაფიტებული (Store, Dept) კომბინაციებისთვის.


In [8]:
import cloudpickle

test_dates = pd.DatetimeIndex(np.sort(test['Date'].unique()), name='Date')
PRED_LEN_TEST = len(test_dates)
assert (test_dates[0] - wide.index.max()).days == 7, 'test must start right after train'
print('test horizon:', PRED_LEN_TEST, 'weeks')


class ARIMAForecastPipeline:
    def __init__(self, forecast_long, sd_mean, dept_mean, global_mean):
        self.forecast_long = forecast_long
        self.sd_mean = sd_mean
        self.dept_mean = dept_mean
        self.global_mean = global_mean

    def predict(self, model_input):
        df = model_input.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        out = (df.merge(self.forecast_long, on=['Store', 'Dept', 'Date'], how='left')
                 .merge(self.sd_mean, on=['Store', 'Dept'], how='left')
                 .merge(self.dept_mean, on='Dept', how='left'))
        pred = (out['pred'].fillna(out['SD_Mean'])
                           .fillna(out['Dept_Mean'])
                           .fillna(self.global_mean))
        return pred.clip(lower=0).values


run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='ARIMA_Final_Pipeline',
                 job_type='final_pipeline', config={**BEST_CFG, 'pred_len': PRED_LEN_TEST})
run.summary['val_wmae'] = best_val  # შეფასება candidate-search-ის holdout-იდან

final_preds, n_fallback_final = fit_forecast_all_series(best_order, wide, PRED_LEN_TEST)
run.log({'n_fallback_series_final': n_fallback_final})

fc = pd.DataFrame(final_preds, index=test_dates, columns=wide.columns)
forecast_long = fc.stack(level=[0, 1], future_stack=True).rename('pred').reset_index()

sd_mean = (train.groupby(['Store', 'Dept'])['Weekly_Sales']
                .mean().rename('SD_Mean').reset_index())
dept_mean = (train.groupby('Dept')['Weekly_Sales']
                  .mean().rename('Dept_Mean').reset_index())

pipeline = ARIMAForecastPipeline(forecast_long, sd_mean, dept_mean,
                                 float(train['Weekly_Sales'].mean()))

with open('arima_pipeline.pkl', 'wb') as f:
    cloudpickle.dump(pipeline, f)

artifact = wandb.Artifact('arima_pipeline', type='model',
                          description='ARIMA end-to-end pipeline: raw test rows -> predictions',
                          metadata={'val_wmae': float(best_val), **BEST_CFG})
artifact.add_file('arima_pipeline.pkl')
run.log_artifact(artifact)

test_pred = pipeline.predict(test[RAW_COLS])
submission = pd.DataFrame({
    'Id': (test['Store'].astype(str) + '_' + test['Dept'].astype(str)
           + '_' + test['Date'].dt.strftime('%Y-%m-%d')),
    'Weekly_Sales': test_pred,
})
submission.to_csv('submission_arima.csv', index=False)
sub_art = wandb.Artifact('arima_submission', type='submission')
sub_art.add_file('submission_arima.csv')
run.log_artifact(sub_art)

val_score = best_val
run.finish()
submission.head()

test horizon: 39 weeks


n_fallback_series_final,▁
n_fallback_series_final,0
val_wmae,1914.19482


,Id,Weekly_Sales
0,1_1_2012-11-02,27481.090782
1,1_1_2012-11-09,27481.090782
2,1_1_2012-11-16,27481.090782
3,1_1_2012-11-23,27481.090782
4,1_1_2012-11-30,27481.090782


In [9]:
import json, os

reg_path = '/content/drive/MyDrive/ML Final/best_runs.json'
info = json.load(open(reg_path)) if os.path.exists(reg_path) else {}
info['ARIMA'] = {
    'artifact': f'{WANDB_PROJECT}/arima_pipeline:latest',
    'val_wmae': float(val_score),
}
with open(reg_path, 'w') as f:
    json.dump(info, f, indent=2)
info

{'XGBoost': {'artifact': 'ML-Final/xgboost_pipeline:latest',
  'val_wmae': 1451.361784955096},
 'LightGBM': {'artifact': 'ML-Final/lightgbm_pipeline:latest',
  'val_wmae': 1508.4106169389534},
 'DLinear': {'artifact': 'ML-Final/dlinear_pipeline:latest',
  'val_wmae': 1602.6865936618267},
 'NBEATS': {'artifact': 'ML-Final/nbeats_pipeline:latest',
  'val_wmae': 1476.8615020275404},
 'PatchTST': {'artifact': 'ML-Final/patchtst_pipeline:latest',
  'val_wmae': 1554.1872775576571},
 'Prophet': {'artifact': 'ML-Final/prophet_pipeline:latest',
  'val_wmae': 1831.404616769137},
 'ARIMA': {'artifact': 'ML-Final/arima_pipeline:latest',
  'val_wmae': 1914.1948218719908}}

In [10]:
import cloudpickle, os

api = wandb.Api()
art = api.artifact(f'{WANDB_PROJECT}/arima_pipeline:latest')
art_dir = art.download()
with open(os.path.join(art_dir, 'arima_pipeline.pkl'), 'rb') as f:
    loaded = cloudpickle.load(f)
print(loaded.predict(test[RAW_COLS].head())[:5])

wandb:   1 of 1 files downloaded.  


[27481.0907818 27481.0907818 27481.0907818 27481.0907818 27481.0907818]
